In [3]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import os

# Konfigurasi Geocoder
geolocator = Nominatim(user_agent="thesis_address_normalizer_v1")
# Tambahkan delay 1.5 detik antar request agar aman
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

In [5]:
# Load data jalan yang rumpang
df_streets = pd.read_csv('../data/processed/jakarta_streets_cleaned.csv') # 
# Load data master wilayah untuk validasi (Opsional tapi disarankan)
df_master_wilayah = pd.read_csv('../data/raw/full2.csv') # 

print(f"Total data jalan: {len(df_streets)}")

Total data jalan: 4192


In [6]:
def get_detailed_location(row):
    # Susun query pencarian
    query = f"{row['NAMA_JALAN']}, {row['KECAMATAN']}, {row['WILAYAH']}, DKI Jakarta"
    
    try:
        # Cari di OpenStreetMap
        location = geocode(query, addressdetails=True)
        
        if location:
            address = location.raw.get('address', {})
            # Ambil kelurahan (OSM menggunakan label berbeda-beda: village, suburb, quarter)
            kelurahan = address.get('village') or address.get('suburb') or address.get('quarter') or address.get('neighbourhood')
            postcode = address.get('postcode')
            
            return pd.Series([kelurahan, postcode, "OSM_Success"])
    except Exception as e:
        return pd.Series([None, None, f"Error: {str(e)}"])
    
    return pd.Series([None, None, "Not_Found"])

# Kita coba test 5 baris pertama dulu sebelum running semua
print("Testing enrichment pada 5 data pertama...")
test_df = df_streets.head(5).copy()
test_df[['KELURAHAN_REF', 'KODEPOS_REF', 'STATUS']] = test_df.apply(get_detailed_location, axis=1)
print(test_df)

Testing enrichment pada 5 data pertama...
         WILAYAH    KECAMATAN                NAMA_JALAN    KELURAHAN_REF  \
0  JAKARTA PUSAT  TANAH ABANG          JL. ADMINISTRASI       Petamburan   
1  JAKARTA PUSAT  TANAH ABANG    JL. ARTERI PEJOMPONGAN             None   
2  JAKARTA PUSAT  TANAH ABANG  JL. ASIA AFRIKA/PINTU IX             None   
3  JAKARTA PUSAT  TANAH ABANG  JL. BENDUNGAN HILIR RAYA  Bendungan Hilir   
4  JAKARTA PUSAT  TANAH ABANG       JL. BENDUNGAN HILIR  Bendungan Hilir   

  KODEPOS_REF       STATUS  
0       10260  OSM_Success  
1        None    Not_Found  
2        None    Not_Found  
3       10210  OSM_Success  
4       10210  OSM_Success  


In [7]:
output_file = '../data/processed/enriched_jakarta_streets.csv'
batch_size = 100 # Simpan setiap 100 data

# Jika file sudah ada, lanjut dari baris terakhir
if os.path.exists(output_file):
    df_processed = pd.read_csv(output_file)
    start_idx = len(df_processed)
    print(f"Melanjutkan dari index: {start_idx}")
else:
    df_processed = pd.DataFrame()
    start_idx = 0

# Proses loop
for i in tqdm(range(start_idx, len(df_streets), batch_size)):
    end_idx = min(i + batch_size, len(df_streets))
    batch_df = df_streets.iloc[i:end_idx].copy()
    
    # Jalankan pencarian
    batch_df[['KELURAHAN_REF', 'KODEPOS_REF', 'STATUS']] = batch_df.apply(get_detailed_location, axis=1)
    
    # Gabungkan dan simpan
    df_processed = pd.concat([df_processed, batch_df], ignore_index=True)
    df_processed.to_csv(output_file, index=False)

print(f"Enrichment selesai! Hasil disimpan di {output_file}")

 67%|██████▋   | 28/42 [1:10:22<35:24, 151.78s/it]RateLimiter caught an error, retrying (0/2 tries). Called with (*('JL. WALET PERMAI 1, PENJARINGAN, JAKARTA UTARA, DKI Jakarta',), **{'addressdetails': True}).
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/http/client.py", line 1395, in getresponse
    response.begin()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/http/client.py", line 323, in begin
    version, status, reason = self._read_status()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/http/client.py", line 284, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859

Enrichment selesai! Hasil disimpan di ../data/processed/enriched_jakarta_streets.csv


In [10]:
import pandas as pd
import re

# 1. Load data
df = pd.read_csv('../data/processed/enriched_jakarta_streets.csv')

# 2. Fungsi yang sudah diperbaiki untuk handle data non-string (NaN)
def get_root_street(street_name):
    # Cek jika street_name bukan string (misal NaN atau float)
    if not isinstance(street_name, str):
        return ""
        
    # Hilangkan angka romawi atau angka di akhir
    root = re.sub(r'\s+([IVXLCDM]+|\d+)$', '', street_name.strip())
    return root

# 3. Pastikan kolom NAMA_JALAN adalah string dan buat root_name
df['root_name'] = df['NAMA_JALAN'].astype(str).apply(get_root_street)

# 4. Buat mapping dari data yang sukses (pastikan tidak memetakan root_name yang kosong)
mapping_data = df[(df['STATUS'] == 'OSM_Success') & (df['root_name'] != "")].drop_duplicates(['root_name', 'KECAMATAN'])
mapping_dict = mapping_data.set_index(['root_name', 'KECAMATAN'])[['KELURAHAN_REF', 'KODEPOS_REF']].to_dict('index')

# 5. Isi data yang kosong (Imputation)
def fill_missing(row):
    # Hanya imputasi jika data aslinya memang tidak ketemu atau rumpang
    if row['STATUS'] != 'OSM_Success':
        key = (row['root_name'], row['KECAMATAN'])
        if key in mapping_dict:
            row['KELURAHAN_REF'] = mapping_dict[key]['KELURAHAN_REF']
            row['KODEPOS_REF'] = mapping_dict[key]['KODEPOS_REF']
            row['STATUS'] = 'Imputed_Success'
    return row

df = df.apply(fill_missing, axis=1)

# 6. Simpan hasil akhir
df.drop(columns=['root_name']).to_csv('../data/processed/master_data_final.csv', index=False)
print("Imputasi selesai! Data rumpang karena nomor jalan sudah berhasil ditambal.")

Imputasi selesai! Data rumpang karena nomor jalan sudah berhasil ditambal.


In [12]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import os
import time

# 1. Konfigurasi
INPUT_FILE = '../data/processed/all_wilayah.csv'
OUTPUT_FILE = '../data/processed/master_complexes_final.csv'
USER_AGENT = "thesis_final_run_landmark" # Ganti tiap kali running baru jika perlu

geolocator = Nominatim(user_agent=USER_AGENT)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# 2. Load Data & Cek Progress Sebelumnya (Checkpoint)
if os.path.exists(OUTPUT_FILE):
    df_complex = pd.read_csv(OUTPUT_FILE)
    print(f"✅ Melanjutkan progress yang ada. Sudah terproses: {len(df_complex[df_complex['STATUS'].notna()])} baris.")
else:
    df_complex = pd.read_csv(INPUT_FILE)
    # Siapkan kolom baru
    for col in ['KELURAHAN_REF', 'KECAMATAN_REF', 'KODEPOS_REF', 'STATUS']:
        df_complex[col] = None
    print(f"📂 Memulai dari awal. Total data: {len(df_complex)}")

# 3. Fungsi Inti
def get_osm_data(name, wilayah):
    # Handle wilayah yang None (seperti di kategori universitas)
    query_wilayah = wilayah if pd.notna(wilayah) else "DKI Jakarta"
    query = f"{name}, {query_wilayah}, DKI Jakarta"
    
    try:
        location = geocode(query, addressdetails=True)
        if location:
            addr = location.raw.get('address', {})
            kel = addr.get('village') or addr.get('suburb') or addr.get('quarter') or addr.get('neighbourhood')
            kec = addr.get('city_district') or addr.get('county') or addr.get('town')
            post = addr.get('postcode')
            return kel, kec, post, "OSM_Success"
    except Exception as e:
        return None, None, None, f"Error: {str(e)}"
    
    return None, None, None, "Not_Found"

# 4. Looping dengan Auto-Save
print("🚀 Memulai proses enrichment. Silakan ditinggal ngopi, ini butuh waktu ~1 jam.")

batch_save_size = 50 # Simpan file setiap 50 data agar aman

for i in tqdm(range(len(df_complex))):
    # Skip jika sudah pernah sukses/not found (untuk checkpoint)
    if pd.notna(df_complex.at[i, 'STATUS']):
        continue
        
    kel, kec, post, status = get_osm_data(df_complex.at[i, 'name'], df_complex.at[i, 'wilayah'])
    
    df_complex.at[i, 'KELURAHAN_REF'] = kel
    df_complex.at[i, 'KECAMATAN_REF'] = kec
    df_complex.at[i, 'KODEPOS_REF'] = post
    df_complex.at[i, 'STATUS'] = status
    
    # Auto-save setiap batch
    if i % batch_save_size == 0:
        df_complex.to_csv(OUTPUT_FILE, index=False)

# Simpan hasil akhir
df_complex.to_csv(OUTPUT_FILE, index=False)
print(f"✨ SELESAI! Data disimpan di: {OUTPUT_FILE}")

📂 Memulai dari awal. Total data: 2718
🚀 Memulai proses enrichment. Silakan ditinggal ngopi, ini butuh waktu ~1 jam.


 83%|████████▎ | 2262/2718 [56:44<11:42,  1.54s/it] RateLimiter caught an error, retrying (0/2 tries). Called with (*('SMK YPIA Al Falah Jakarta, Jakarta Timur, DKI Jakarta',), **{'addressdetails': True}).
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/http/client.py", line 1395, in getresponse
    response.begin()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/http/client.py", line 323, in begin
    version, status, reason = self._read_status()
  File "/opt/homebrew/anaconda3/envs/thesis/lib/python3.10/http/client.py", line 284, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")

✨ SELESAI! Data disimpan di: ../data/processed/master_complexes_final.csv


In [13]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import os
import re

# 1. Setup Geocoder
geolocator = Nominatim(user_agent="thesis_second_chance_v4")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# 2. Load Data yang sebelumnya sudah kamu simpan
path_file = '../data/processed/master_complexes_final.csv'
df = pd.read_csv(path_file)

def clean_query_name(name):
    # Hapus noise seperti 'Jakarta Pusat', 'Jaktim' dll yang nyelip di nama
    noise = ['JAKARTA PUSAT', 'JAKARTA SELATAN', 'JAKARTA TIMUR', 'JAKARTA BARAT', 'JAKARTA UTARA', 'JAKARTA']
    for n in noise:
        name = re.sub(n, '', name, flags=re.IGNORECASE)
    return name.strip()

def get_details_relaxed(name, wilayah):
    # Tahap 1: Query Standar (Gedung + Jakarta)
    clean_name = clean_query_name(name)
    queries = [
        f"{clean_name}, {wilayah}, Jakarta", # Lebih spesifik
        f"{clean_name}, Jakarta"             # Lebih santai
    ]
    
    # Tahap 2: Jika mengandung kata kunci tertentu, coba hapus prefiksnya
    # Misal: "Apartemen Sudirman" -> "Sudirman"
    prefixes = ['APARTEMEN', 'WISMA', 'GEDUNG', 'RS', 'RUMAH SAKIT', 'HOTEL']
    for p in prefixes:
        if clean_name.upper().startswith(p):
            short_name = re.sub(f"^{p}", "", clean_name, flags=re.IGNORECASE).strip()
            queries.append(f"{short_name}, Jakarta")

    # Eksekusi pencarian berjenjang
    for q in queries:
        try:
            location = geocode(q, addressdetails=True)
            if location:
                addr = location.raw.get('address', {})
                kel = addr.get('village') or addr.get('suburb') or addr.get('quarter') or addr.get('neighbourhood')
                kec = addr.get('city_district') or addr.get('county') or addr.get('town')
                post = addr.get('postcode')
                return kel, kec, post, "OSM_Success"
        except:
            continue
    return None, None, None, "Not_Found"

# 3. Hanya proses yang masih 'Not_Found'
not_found_mask = df['STATUS'] == 'Not_Found'
df_to_fix = df[not_found_mask]

print(f"Mencoba memperbaiki {len(df_to_fix)} data yang Not_Found...")

for i in tqdm(df_to_fix.index):
    kel, kec, post, status = get_details_relaxed(df.at[i, 'name'], df.at[i, 'wilayah'])
    
    if status == "OSM_Success":
        df.at[i, 'KELURAHAN_REF'] = kel
        df.at[i, 'KECAMATAN_REF'] = kec
        df.at[i, 'KODEPOS_REF'] = post
        df.at[i, 'STATUS'] = "OSM_Success_Recovered"

# 4. Simpan kembali
df.to_csv('../data/processed/master_complexes_final_v2.csv', index=False)

success_now = len(df[df['STATUS'].str.contains('Success')])
print(f"\n✨ Selesai! Sekarang total {success_now} dari {len(df)} data berhasil ditemukan.")

Mencoba memperbaiki 1385 data yang Not_Found...


  0%|          | 3/1385 [00:07<1:02:56,  2.73s/it]/var/folders/8l/thmsb_394sv83fjwddsbbf340000gn/T/ipykernel_8420/1881012981.py:65: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '14410' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, 'KODEPOS_REF'] = post
100%|██████████| 1385/1385 [1:10:22<00:00,  3.05s/it]


✨ Selesai! Sekarang total 1498 dari 2718 data berhasil ditemukan.
